# LLM-as-judge — rubric, pairwise, position-swap

Three judge patterns from cheapest to most reliable:

1. **Rubric** — score one answer on a 0–4 scale. Cheap. Suffers from miscalibration.
2. **Pairwise** — pick the better of two answers. More reliable but has *position bias*.
3. **Position-swap** — ask twice with answers flipped; only count a win if it survives the swap.

We run all three on a committed 5-pair fixture so the notebook is fully offline.

In [ ]:
import os, sys, pathlib, json
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
leaf = pathlib.Path.cwd()
sys.path.insert(0, str(leaf))
from eval import rubric_score, pairwise, pairwise_swapped
PAIRS = json.loads((leaf / 'pairs.json').read_text(encoding='utf-8'))
for p in PAIRS:
    print(f"{p['id']}: {p['question']}")

## Rubric scoring — absolute 0–4

In [ ]:
print(f'{"id":<5}{"rubric_a":>10}{"rubric_b":>10}')
for p in PAIRS:
    ra = rubric_score(p['question'], p['answer_a'])
    rb = rubric_score(p['question'], p['answer_b'])
    print(f'{p["id"]:<5}{ra:>10}{rb:>10}')

## Pairwise — raw vs swap-mitigated

`pairwise_swapped` calls the judge twice with answers flipped; only counts a win that survives the swap. The **delta** between raw and swapped A-win-rate is the position-bias signal you actually care about — if it's > 0.1, your judge is fundamentally unreliable for this task.

In [ ]:
raw = {'A': 0, 'B': 0, 'tie': 0}
swap = {'A': 0, 'B': 0, 'tie': 0}
for p in PAIRS:
    r = pairwise(p['question'], p['answer_a'], p['answer_b']);  raw[r] += 1
    s = pairwise_swapped(p['question'], p['answer_a'], p['answer_b']);  swap[s] += 1
n = len(PAIRS)
print(f"raw      A-win rate: {raw['A']/n:.2f}   B-win: {raw['B']/n:.2f}   tie: {raw['tie']/n:.2f}")
print(f"swapped  A-win rate: {swap['A']/n:.2f}   B-win: {swap['B']/n:.2f}   tie: {swap['tie']/n:.2f}")
print(f"position-bias delta (raw_A - swap_A) = {(raw['A']-swap['A'])/n:+.2f}")

## Calibration check — agreement between rubric and pairwise

If rubric_a > rubric_b but pairwise picks B (or vice versa), your two judges disagree. Track this disagreement rate as a sanity check on the judges themselves.

In [ ]:
disagreements = 0
for p in PAIRS:
    ra = rubric_score(p['question'], p['answer_a'])
    rb = rubric_score(p['question'], p['answer_b'])
    pw = pairwise_swapped(p['question'], p['answer_a'], p['answer_b'])
    rubric_pick = 'A' if ra > rb else 'B' if rb > ra else 'tie'
    if rubric_pick != pw:
        disagreements += 1
print(f'rubric vs pairwise disagreement: {disagreements}/{len(PAIRS)}')

## Takeaways

* Always use **pairwise + position-swap** for headline comparisons; rubric is fine for tracking absolute drift over time.
* Track the **position-bias delta** and the **rubric/pairwise disagreement rate** as judge-health KPIs.
* If position-bias delta > 0.1 on a 50–100 pair sample, switch judge models or shrink the rubric (4 → 3 levels).